# Capítulo 27 · Laboratório 27: sistemas, backtesting e aprendizado de máquina
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026** (40 ações desde 2005, Ibovespa desde 1999 e CDI do Banco Central). Na primeira execução, o notebook baixa os dados do Yahoo Finance e do Banco Central na sua máquina e guarda uma cópia local (`dados_cap27.pkl`), que as execuções seguintes reaproveitam. O Yahoo corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Regras públicas, custo de 0,10% ida e volta. A Parte 10 (aprendizado de máquina) leva cerca de um minuto. Os sorteios usam semente fixa, com um gerador próprio em cada Parte: os números se repetem a cada execução, mesmo quando uma Parte é rodada de novo sozinha.

In [ ]:
!pip install -q yfinance

In [ ]:
import io
import logging
import re
import os
import time
import urllib.request
from math import comb
import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
FIM, CUSTO, CORTE = "2026-10-01", 0.001, "2016-01-01"
CACHE = "dados_cap27.pkl"


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


def mostra(*partes):
    """print com vírgula decimal e ponto de milhar (padrão brasileiro)."""
    texto = " ".join(str(p) for p in partes)
    texto = re.sub(r"(?<=\d)\.(?=\d)", ",", texto)
    print(re.sub(r"(?<=\d)_(?=\d)", ".", texto))


def rsi(c, n):
    d = c.diff()
    return 100 - 100 / (1 + d.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
                        / (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean())


def limpa(d):
    d = d.dropna()
    return d[(d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])]


def limpa_estrita(d):
    d = d.dropna()
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"]) & (d["Volume"] > 0)
    return d[ok]


def sharpe(r):
    r = r.dropna()
    return np.sqrt(252) * r.mean() / r.std() if r.std() > 0 else 0.0


def cruzamento(c, rapida, lenta, caixa=None):
    """Comprado quando a média simples rápida está acima da lenta (sinal de ontem); fora, caixa (ou zero)."""
    pos = (c.rolling(rapida).mean() > c.rolling(lenta).mean()).astype(float).shift(1).fillna(0)
    ret = c.pct_change().fillna(0)
    fora = caixa.reindex(c.index).fillna(0) if caixa is not None else 0.0
    return pos * ret + (1 - pos) * fora - pos.diff().abs().fillna(0) * CUSTO / 2


def ops_ifr2(d, sobrepor=True):
    """IFR(2) de Connors: acima da MM200, IFR(2) < 10, compra no fechamento, sai ao fechar acima da MM5 (até 10 dias).
    sobrepor=True conta cada sinal (como no Cap. 27); False opera uma posição por vez (curva de capital real)."""
    c = d["Close"]
    C, H, L = c.to_numpy(), d["High"].to_numpy(), d["Low"].to_numpy()
    filtro = (c > c.rolling(200).mean()).to_numpy()
    sinal = filtro & (rsi(c, 2) < 10).to_numpy()
    sai = (c > c.rolling(5).mean()).to_numpy()
    out, livre = [], 0
    for i in np.flatnonzero(sinal):
        if i < 201 or i > len(C) - 12 or (not sobrepor and i < livre):
            continue                                  # sem sobrepor: só entra depois de sair da anterior
        k = next((k for k in range(i + 1, i + 11) if sai[k]), i + 10)
        livre = k + 1
        out.append({"i": i, "k": k, "r": C[k] / C[i] - 1 - CUSTO, "dias": k - i,
                    "mfe": H[i + 1:k + 1].max() / C[i] - 1, "mae": L[i + 1:k + 1].min() / C[i] - 1})
    return pd.DataFrame(out), filtro

## Parte 1: dados

In [ ]:
if os.path.exists(CACHE):              # guarda o download: o provedor pode corrigir o histórico entre execuções
    dados, ib, cdi_bruto = pd.read_pickle(CACHE)
else:
    dados = yf.download(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, progress=False, group_by="ticker")
    ib = yf.download("^BVSP", start="1999-01-01", end=FIM, auto_adjust=True, progress=False, multi_level_index=False)
    confere({"as 40 ações": dados, "^BVSP": ib})
    url = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.4391/dados?formato=csv&dataInicial=01/01/1999"
           "&dataFinal=30/09/2026")
    for tentativa in range(5):                              # a API do Banco Central às vezes falha: tenta de novo
        try:
            cdi_bruto = urllib.request.urlopen(url, timeout=60).read().decode("latin-1")
            break
        except Exception:
            time.sleep(5 * (tentativa + 1))
    else:
        raise RuntimeError("Não consegui baixar o CDI do Banco Central depois de 5 tentativas. O site pode estar fora do"
                           " ar. Espere alguns minutos e rode a célula de novo.")
    pd.to_pickle((dados, ib, cdi_bruto), CACHE)
D = {t: limpa(dados[t]) for t in TICKERS}
D["SUZB3.SA"] = D["SUZB3.SA"].loc["2017-11-10":]          # antes disso, preço parado no Yahoo (Capítulo 26)
BASE = {t: limpa_estrita(dados[t]) for t in TICKERS}        # sem barras sem negócio (Parte 7)
cdi_m = pd.read_csv(io.StringIO(cdi_bruto), sep=";", decimal=",")
cdi_m.index = pd.to_datetime(cdi_m["data"], dayfirst=True).dt.to_period("M")
cdi_m = cdi_m["valor"].astype(float) / 100
mostra(f"Parte 1. {len(TICKERS)} ações desde 2005, Ibovespa desde 1999 e CDI mensal; dados até 30/09/2026")

## Parte 2: otimizar dentro da amostra, conferir fora

In [ ]:
GRADE = [(f, s) for f in (5, 10, 20, 30, 50) for s in (20, 50, 100, 150, 200, 250) if f < s]
dentro, fora_ = {}, {}
for f, s in GRADE:
    sd, sf = [], []
    for t in TICKERS:
        r = cruzamento(D[t]["Close"], f, s)
        sd.append(sharpe(r.loc["2006":"2015"]))
        sf.append(sharpe(r.loc["2016":"2026"]))
    dentro[(f, s)], fora_[(f, s)] = np.mean(sd), np.mean(sf)
G = pd.DataFrame({"dentro": dentro, "fora": fora_})
G["posição dentro"], G["posição fora"] = G["dentro"].rank(ascending=False), G["fora"].rank(ascending=False)
melhor = G["dentro"].idxmax()
rho = stats.spearmanr(G["dentro"], G["fora"]).statistic
mostra(f"Parte 2. Cruzamento de médias, {len(GRADE)} combinações, média do índice de Sharpe nas 40 ações")
mostra(f"   melhor dentro da amostra (2006-2015): {melhor[0]}/{melhor[1]}, Sharpe {G.loc[melhor, 'dentro']:.2f}"
      f" -> fora da amostra (2016-2026): {G.loc[melhor, 'fora']:.2f}, posição {int(G.loc[melhor, 'posição fora'])} de {len(G)}")
mostra(f"   correlação de postos entre o Sharpe de dentro e o de fora: {rho:+.2f}")
mostra(f"   50/200 fixo: dentro {G.loc[(50, 200), 'dentro']:.2f} | fora {G.loc[(50, 200), 'fora']:.2f}")
bh_f = np.mean([sharpe(D[t]["Close"].pct_change().loc["2016":"2026"]) for t in TICKERS])
mostra(f"   comprar e segurar, fora da amostra: {bh_f:.2f}")
# a melhor combinação de cada ação (otimização por ativo)
ganho_d, ganho_f = [], []
for t in TICKERS:
    c = D[t]["Close"]
    res = {(f, s): sharpe(cruzamento(c, f, s).loc["2006":"2015"]) for f, s in GRADE}
    b = max(res, key=res.get)
    ganho_d.append(res[b] - sharpe(cruzamento(c, 50, 200).loc["2006":"2015"]))
    ganho_f.append(sharpe(cruzamento(c, *b).loc["2016":"2026"]) - sharpe(cruzamento(c, 50, 200).loc["2016":"2026"]))
mostra(f"   otimizando ação por ação: vantagem sobre o 50/200 dentro {np.mean(ganho_d):+.2f} de Sharpe,"
      f" fora {np.mean(ganho_f):+.2f}; a otimizada perdeu fora em {np.mean(np.array(ganho_f) < 0):.0%} das ações")

## Parte 3: a superfície dos parâmetros (Kaufman)

In [ ]:
ENT = [5, 8, 12, 18, 27, 40, 60, 90, 135]          # progressão geométrica (Kaufman: passos de 1,5 vez)
SAI = [3, 5, 8, 12, 18, 27, 40, 60]
RET = {t: BASE[t]["Close"].pct_change() for t in TICKERS}


def posicao(d, e, s):
    """Comprado depois de fechar acima da máxima de e dias; zera ao fechar abaixo da mínima de s dias."""
    C = d["Close"].to_numpy()
    hi = d["High"].rolling(e).max().shift(1).to_numpy()
    lo = d["Low"].rolling(s).min().shift(1).to_numpy()
    pos = np.zeros(len(C))
    p = 0.0
    for i in range(len(C)):
        if p == 0 and C[i] > hi[i]:
            p = 1.0
        elif p == 1 and C[i] < lo[i]:
            p = 0.0
        pos[i] = p
    return pd.Series(pos, index=d.index)


GRADE_D, GRADE_F = pd.DataFrame(index=ENT, columns=SAI, dtype=float), pd.DataFrame(index=ENT, columns=SAI, dtype=float)
for e in ENT:
    for s in SAI:
        if s >= e:
            continue
        cart = []
        for t in TICKERS:
            p = posicao(BASE[t], e, s)
            cart.append((p.shift(1) * RET[t] - p.diff().abs() * CUSTO / 2).rename(t))
        port = pd.concat(cart, axis=1, sort=True).loc["2006":].mean(axis=1)      # peso igual entre as ações com dado
        GRADE_D.loc[e, s], GRADE_F.loc[e, s] = sharpe(port[port.index < CORTE]), sharpe(port[port.index >= CORTE])


def suaviza(g):
    """Kaufman: troca cada resultado pela média dele com os vizinhos válidos (até 9 casas)."""
    m = g.to_numpy(dtype=float)
    out = np.full_like(m, np.nan)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            if np.isnan(m[i, j]):
                continue
            viz = m[max(i - 1, 0):i + 2, max(j - 1, 0):j + 2]
            out[i, j] = np.nanmean(viz)
    return pd.DataFrame(out, index=g.index, columns=g.columns)


SUAVE = suaviza(GRADE_D)
pico = GRADE_D.stack().dropna().idxmax()
plato = SUAVE.stack().dropna().idxmax()
pares = pd.DataFrame({"dentro": GRADE_D.stack(), "fora": GRADE_F.stack()}).dropna()
validos = pares.index
pos_fora = pares["fora"].rank(ascending=False)
mostra(f"\nParte 3. Rompimento de Donchian só comprado nas 40 ações (peso igual), {len(validos)} combinações; índice de Sharpe")
mostra(f"   dentro da amostra (2006 a 2015) x fora (2016 a 2026)")
mostra(f"   pico de dentro: entra {pico[0]} / sai {pico[1]} dias -> dentro {GRADE_D.loc[pico]:.2f} | fora {GRADE_F.loc[pico]:.2f}"
      f" ({int(pos_fora[pico])}º de {len(validos)} fora)")
mostra(f"   centro do platô (média com os vizinhos): entra {plato[0]} / sai {plato[1]} dias -> dentro {GRADE_D.loc[plato]:.2f}"
      f" | fora {GRADE_F.loc[plato]:.2f} ({int(pos_fora[plato])}º de {len(validos)} fora)")
mostra(f"   média de todos os testes (a expectativa de Kaufman): dentro {pares['dentro'].mean():.2f} | fora {pares['fora'].mean():.2f};"
      f" média menos 1 desvio: dentro {pares['dentro'].mean() - pares['dentro'].std():.2f}")
mostra(f"   testes com Sharpe acima de zero (robustez de Kaufman): dentro {np.mean(pares['dentro'] > 0):.0%} | fora {np.mean(pares['fora'] > 0):.0%}")
mostra(f"   correlação de postos entre dentro e fora: {pares['dentro'].rank().corr(pares['fora'].rank()):+.2f}")
mostra(f"   tartarugas (20/10 e 55/20 mais próximos na grade: 18/8 e 60/18): fora {GRADE_F.loc[18, 8]:.2f} e {GRADE_F.loc[60, 18]:.2f}")
bh = pd.concat([RET[t].rename(t) for t in TICKERS], axis=1, sort=True).loc["2006":].mean(axis=1)
mostra(f"   régua: comprar e segurar as 40 ações com peso igual -> dentro {sharpe(bh[bh.index < CORTE]):.2f} | fora {sharpe(bh[bh.index >= CORTE]):.2f}")
mostra(f"   testes que bateram a régua: dentro {np.mean(pares['dentro'] > sharpe(bh[bh.index < CORTE])):.0%} | fora {np.mean(pares['fora'] > sharpe(bh[bh.index >= CORTE])):.0%}")

## Parte 4: walk-forward no Ibovespa

In [ ]:
ibc = ib["Close"].dropna()
dias_mes = ibc.groupby(ibc.index.to_period("M")).transform("size")
cdi_d = pd.Series((1 + cdi_m.reindex(ibc.index.to_period("M")).to_numpy()) ** (1 / dias_mes.to_numpy()) - 1, index=ibc.index).fillna(0)
partes = []
for ano in range(2010, 2027):
    treino = ibc.loc[str(ano - 5):str(ano - 1)]
    b = max(GRADE, key=lambda p: sharpe(cruzamento(ibc, *p, caixa=cdi_d).loc[treino.index]))
    partes.append(cruzamento(ibc, *b, caixa=cdi_d).loc[str(ano)])
wf = pd.concat(partes)
fixo = cruzamento(ibc, 50, 200, caixa=cdi_d).loc[wf.index]
bh = ibc.pct_change().loc[wf.index]
anos = len(wf) / 252


def pct(x, k=1):
    """Com sinal; o que arredonda para zero sai sem sinal (0,0%)."""
    return f"{0.0:.{k}%}" if round(x * 100, k) == 0 else f"{x:+.{k}%}"


mostra("\nParte 4. Walk-forward no Ibovespa, 2010 a 30/09/2026 (fora do mercado = CDI)")
for rot, r in [("walk-forward (reotimiza todo ano)", wf), ("50/200 fixo", fixo), ("comprar e segurar", bh),
               ("só CDI", cdi_d.loc[wf.index])]:
    eq = (1 + r).cumprod()
    mostra(f"   {rot:<34} {pct(eq.iloc[-1] ** (1 / anos) - 1)} ao ano | maior queda {(eq / eq.cummax() - 1).min():.0%}")

## Parte 5: hipótese nula com entradas aleatórias

In [ ]:
reais, pools = [], []
for t in TICKERS:
    o, filtro = ops_ifr2(D[t])
    if len(o) == 0:
        continue
    reais.append(o)
    C = D[t]["Close"].to_numpy()
    elegiveis = np.flatnonzero(filtro[201:len(C) - 12]) + 201
    pools.append((C, elegiveis, o["dias"].to_numpy()))
R = pd.concat(reais)
media_real = R["r"].mean()
rng = np.random.default_rng(29)          # gerador próprio da Parte 5, com semente fixa
medias = []
for _ in range(1000):
    s = []
    for C, eleg, dias in pools:
        ii = rng.choice(eleg, size=len(dias))
        s.append(C[ii + dias] / C[ii] - 1 - CUSTO)
    medias.append(np.concatenate(s).mean())
medias = np.array(medias)
mostra(f"\nParte 5. IFR(2) de Connors ({len(R):_} operações) contra entradas aleatórias acima da MM200, mesmo prazo, 1_000 sorteios")
mostra(f"   IFR(2): {media_real:+.2%} por operação | aleatório: média {medias.mean():+.2%}, 99% dos sorteios até"
      f" {np.percentile(medias, 99):+.2%} | sorteios iguais ou melhores que o IFR(2): {np.mean(medias >= media_real):.1%}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.hist(medias * 100, bins=40, color="0.75", edgecolor="0.4", label="1.000 sorteios de entradas aleatórias")
plt.axvline(media_real * 100, color="black", label="IFR(2) de Connors")
plt.xlabel("resultado médio por operação, com custo (%)")
plt.legend()
plt.show()

## Parte 6: quantas operações bastam

In [ ]:
mostra("Parte 6. Quantas operações para a vantagem aparecer (t perto de 2; desvio de 1 R por operação)")
for edge in (0.05, 0.10, 0.20, 0.30):
    mostra(f"   expectativa de {edge:.2f} R por operação: cerca de {int(np.ceil((2 / edge) ** 2)):_} operações")
for s in (0.3, 0.5, 1.0):
    mostra(f"   carteira com índice de Sharpe {s}: cerca de {(2 / s) ** 2:.0f} anos")


rng = np.random.default_rng(2906)        # gerador próprio da Parte 6, com semente fixa
x = R["r"].to_numpy()
mostra(f"\nParte 6. Incerteza da média por operação (desvio das operações do IFR(2): {x.std():.2%})")
for n in (30, 50, 100, 500, 1000):
    ep = x.std() / np.sqrt(n)
    zero = x - x.mean()                                     # mesmas operações, vantagem zerada
    sorte = np.mean([rng.choice(zero, n).mean() >= media_real for _ in range(5000)])
    mostra(f"   {n:>5} operações: média ± {1.96 * ep:.2%} (95%) | sem vantagem nenhuma, chance de mostrar"
          f" {media_real:+.2%} ou mais: {sorte:.0%}")

## Parte 7: sorte ou habilidade

In [ ]:
rng_s = np.random.default_rng(291)       # gerador próprio da Parte 7, com semente fixa
p8 = sum(comb(10, k) for k in range(8, 11)) / 2 ** 10
sims = rng_s.binomial(10, 0.5, size=(10_000, 1000))
mostra(f"\nParte 7. Mil gestores sem habilidade nenhuma, 10 anos, 50% de chance por ano de bater o índice:")
mostra(f"   chance de bater em 8 ou mais anos: {p8:.2%}; esperados em mil: {1000 * p8:.0f} (simulação: {np.mean((sims >= 8).sum(1)):.1f});"
      f" pelo menos um com 10 em 10: {1 - (1 - 0.5 ** 10) ** 1000:.0%}")

lin = []
for t in TICKERS:
    d = BASE[t]
    c = d["Close"]
    C = c.to_numpy()
    sinal = ((c > c.rolling(200).mean()) & (rsi(c, 2) < 10)).to_numpy()
    sai = (c > c.rolling(5).mean()).to_numpy()
    res, livre = [], 0
    for i in np.flatnonzero(sinal):
        if i < 201 or i > len(C) - 12 or i < livre:
            continue
        k = next((k for k in range(i + 1, i + 11) if sai[k]), i + 10)
        res.append(C[k] / C[i] - 1 - CUSTO)
        livre = k + 1
    res = np.array(res)
    lin.append((t, len(res), np.mean(res > 0)))
A = pd.DataFrame(lin, columns=["ativo", "n", "acerto"])
p_med = np.average(A["acerto"], weights=A["n"])
var_obs = A["acerto"].var(ddof=1)
var_sorte = np.mean(p_med * (1 - p_med) / A["n"])
mostra(f"\nParte 7. IFR(2) de Connors nas 40 ações (uma posição por vez): {A['n'].sum():_} operações, {A['n'].median():.0f} por ação (mediana)")
mostra(f"   acerto médio {p_med:.1%}; acerto por ação de {A['acerto'].min():.0%} a {A['acerto'].max():.0%}")
mostra(f"   variância observada {var_obs:.5f} | variância que a sorte sozinha daria {var_sorte:.5f}"
      f" | parcela da sorte {min(var_sorte / var_obs, 1):.0%}")

anual = pd.DataFrame({t: BASE[t]["Close"].resample("YE").last().pct_change() for t in TICKERS}).loc["2006":"2025"]
q_seg, corr = [], []
for a in range(len(anual) - 1):
    x, y = anual.iloc[a].dropna(), anual.iloc[a + 1]
    y = y[x.index].dropna()
    x = x[y.index]
    if len(x) < 20:
        continue
    top, bot = x >= x.quantile(0.8), x <= x.quantile(0.2)
    q_seg.append((x[top].mean() - x[bot].mean(), y[top].mean() - y[bot].mean()))
    corr.append(x.rank().corr(y.rank()))
q_seg = np.array(q_seg)
mostra(f"\nParte 7. Retorno anual das 40 ações, 2006 a 2025: quintil de cima menos quintil de baixo")
mostra(f"   no ano do ranking: {q_seg[:, 0].mean():+.0%} em média; no ano seguinte: {q_seg[:, 1].mean():+.1%}"
      f" (sobrou {q_seg[:, 1].mean() / q_seg[:, 0].mean():.0%} da diferença); o quintil de cima repetiu a vantagem em"
      f" {np.mean(q_seg[:, 1] > 0):.0%} dos anos; correlação de postos de um ano para o outro: {np.mean(corr):+.2f}")
for sd_s in (0.2, 0.5, 1.0, 3.0):
    c = sd_s ** 2 / (sd_s ** 2 + 1)
    skill = rng_s.normal(0, sd_s, 2000)
    y1, y2 = skill + rng_s.normal(0, 1, 2000), skill + rng_s.normal(0, 1, 2000)
    top, bot = y1 >= np.quantile(y1, 0.8), y1 <= np.quantile(y1, 0.2)
    sob = (y2[top].mean() - y2[bot].mean()) / (y1[top].mean() - y1[bot].mean())
    mostra(f"   simulação: habilidade = {c:.0%} da variância -> sobra {sob:.0%} da diferença entre quintis no período seguinte")

## Parte 8: Monte Carlo da ordem das operações

In [ ]:
P, _ = ops_ifr2(D["PETR4.SA"], sobrepor=False)
r = P["r"].to_numpy()


def pior_queda(seq):
    eq = np.cumprod(1 + seq)
    return (eq / np.maximum.accumulate(eq) - 1).min()


def maior_seq_perdas(seq):
    m = a = 0
    for x in seq:
        a = a + 1 if x <= 0 else 0
        m = max(m, a)
    return m


hist_dd, hist_seq = pior_queda(r), maior_seq_perdas(r)
rng = np.random.default_rng(2908)        # gerador próprio da Parte 8, com semente fixa
mc = np.array([(pior_queda(s), maior_seq_perdas(s)) for s in (rng.permutation(r) for _ in range(10000))])
mostra(f"\nParte 8. Monte Carlo: as {len(r)} operações de IFR(2) em PETR4, embaralhadas 10_000 vezes")
mostra(f"   ordem real: pior queda {hist_dd:.0%}, maior sequência de perdas {hist_seq}")
mostra(f"   embaralhadas: pior queda mediana {np.median(mc[:, 0]):.0%}, 5% piores {np.percentile(mc[:, 0], 5):.0%};"
      f" sequência de perdas mediana {np.median(mc[:, 1]):.0f}, 5% piores {np.percentile(mc[:, 1], 95):.0f}")

## Parte 9: o relatório de testes

In [ ]:
ib2 = limpa(ib)
O, Z = ops_ifr2(ib2, sobrepor=False)
O = O.loc[O["i"] >= ib2.index.searchsorted(pd.Timestamp("2000-01-01"))]
g, p_ = O.loc[O["r"] > 0, "r"], O.loc[O["r"] <= 0, "r"]
pos = pd.Series(0.0, index=ib2.index)
for _, o in O.iterrows():
    pos.iloc[int(o["i"]) + 1:int(o["k"]) + 1] = 1.0
ret = ib2["Close"].pct_change().fillna(0)
diario = (pos * ret).loc["2000":]
for _, o in O.iterrows():                                   # custo no dia da saída
    diario.iloc[diario.index.get_loc(ib2.index[int(o["k"])])] -= CUSTO
eq = (1 + diario).cumprod()
sinais = np.sign(O["r"].to_numpy())
n1, n2 = (sinais > 0).sum(), (sinais <= 0).sum()
corridas = 1 + np.sum(sinais[1:] != sinais[:-1])
esp = 2 * n1 * n2 / (n1 + n2) + 1
var = 2 * n1 * n2 * (2 * n1 * n2 - n1 - n2) / ((n1 + n2) ** 2 * (n1 + n2 - 1))
mostra(f"\nParte 9. Relatório: IFR(2) de Connors no Ibovespa, 2000 a 30/09/2026, custo de {CUSTO:.2%} por operação")
linhas_rel = [
    ("operações", f"{len(O)}"), ("acerto", f"{len(g) / len(O):.0%}"),
    ("ganho médio / perda média", f"{g.mean():+.2%} / {p_.mean():+.2%} (payoff {g.mean() / -p_.mean():.2f})"),
    ("fator de lucro", f"{g.sum() / -p_.sum():.2f}"), ("expectativa por operação", f"{O['r'].mean():+.2%}"),
    ("maior sequência de perdas", f"{maior_seq_perdas(O['r'].to_numpy())}"),
    ("tempo posicionado", f"{(pos.loc['2000':] > 0).mean():.0%} dos pregões"),
    ("retorno ao ano / maior queda", f"{eq.iloc[-1] ** (252 / len(eq)) - 1:+.1%} / {(eq / eq.cummax() - 1).min():.0%}"),
    ("Sharpe (sem taxa livre de risco)", f"{sharpe(diario):.2f}"),
    ("MFE médio / MAE médio", f"{O['mfe'].mean():+.2%} / {O['mae'].mean():+.2%}"),
    ("correlação resultado x MFE / x MAE", f"{O['r'].corr(O['mfe']):.2f} / {O['r'].corr(O['mae']):.2f}"),
    ("Z-score (teste de sequências)", f"{(corridas - esp) / np.sqrt(var):+.2f}"),
]
for k, v in linhas_rel:
    mostra(f"   {k:<36} {v}")



def z_corridas(res):
    """Z-score do teste de sequências (corridas) de ganhos e perdas: negativo = sequências; positivo = alternância."""
    sg = np.sign(res)
    a_, b_ = (sg > 0).sum(), (sg <= 0).sum()
    corr_ = 1 + np.sum(sg[1:] != sg[:-1])
    e_ = 2 * a_ * b_ / (a_ + b_) + 1
    v_ = 2 * a_ * b_ * (2 * a_ * b_ - a_ - b_) / ((a_ + b_) ** 2 * (a_ + b_ - 1))
    return (corr_ - e_) / np.sqrt(v_)


S_, _ = ops_ifr2(ib2, sobrepor=True)
S_ = S_.loc[S_["i"] >= ib2.index.searchsorted(pd.Timestamp("2000-01-01"))]
mostra(f"   contando cada sinal (operações sobrepostas): {len(S_)} sinais | {S_['r'].mean():+.2%} por operação |"
      f" Z-score {z_corridas(S_['r'].to_numpy()):+.2f} (sequências falsas: o mesmo movimento conta várias vezes)")

## Parte 10: aprendizado de máquina

In [ ]:
linhas = []
for t in TICKERS:
    d = D[t]
    c, h, l = d["Close"], d["High"], d["Low"]
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    f = pd.DataFrame({
        "ret1": c.pct_change(), "ret5": c.pct_change(5), "ret20": c.pct_change(20), "ret60": c.pct_change(60),
        "dist20": c / c.rolling(20).mean() - 1, "dist200": c / c.rolling(200).mean() - 1,
        "ifr2": rsi(c, 2), "ifr14": rsi(c, 14),
        "dcatr": tr.ewm(alpha=1 / 20, adjust=False).mean() / c.ewm(span=20, adjust=False).mean(),
        "vol20": c.pct_change().rolling(20).std(),
        "faixa": (c - l.rolling(20).min()) / (h.rolling(20).max() - l.rolling(20).min()),
    })
    f["fut5"] = c.shift(-5) / c - 1
    f["alvo"] = (f["fut5"] > 0).astype(int)
    f["ativo"], f["ano"] = t, f.index.year
    linhas.append(f.iloc[200:-5].replace([np.inf, -np.inf], np.nan).dropna())
M = pd.concat(linhas).sort_index()
X_cols = [k for k in M.columns if k not in ("fut5", "alvo", "ativo", "ano")]
modelos = {"regressão logística": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
           "árvores (gradient boosting)": lambda: HistGradientBoostingClassifier(max_iter=200, max_depth=4,
                                                                                min_samples_leaf=500, random_state=0)}
mostra(f"\nParte 10. Prever se a ação sobe nos próximos 5 pregões ({len(M):_} linhas, {len(X_cols)} variáveis técnicas)")
mostra(f"   taxa de base (subiu em 5 pregões): {M['alvo'].mean():.1%}")
for nome, fab in modelos.items():
    acertos, aucs, topo, todos = [], [], [], []
    for ano in range(2012, 2027):
        tr_ = M[(M["ano"] < ano) & (M.index < pd.Timestamp(f"{ano}-01-01") - pd.Timedelta(days=10))]   # purga
        te_ = M[M["ano"] == ano]
        m = fab().fit(tr_[X_cols], tr_["alvo"])
        p = m.predict_proba(te_[X_cols])[:, 1]
        acertos.append(((p > 0.5) == te_["alvo"]).mean())
        aucs.append(roc_auc_score(te_["alvo"], p))
        corte = np.quantile(p, 0.9)
        topo.append(te_["fut5"][p >= corte].mean())
        todos.append(te_["fut5"].mean())
    mostra(f"   {nome:<28} walk-forward 2012-2026: acerto {np.mean(acertos):.1%} | AUC {np.mean(aucs):.3f} |"
          f" 10% mais confiantes {np.mean(topo):+.2%} x todos {np.mean(todos):+.2%} em 5 pregões (sem custo)")
amostra = M.sample(150_000, random_state=0)
kf = KFold(5, shuffle=True, random_state=0)
acc_emb = []
for a, b in kf.split(amostra):
    m = modelos["árvores (gradient boosting)"]().fit(amostra.iloc[a][X_cols], amostra.iloc[a]["alvo"])
    acc_emb.append(roc_auc_score(amostra.iloc[b]["alvo"], m.predict_proba(amostra.iloc[b][X_cols])[:, 1]))
vaz = M.copy()
vaz["ret5_amanha"] = vaz.groupby("ativo")["ret5"].shift(-5)                 # variável que olha o futuro
amostra_v = vaz.dropna().sample(150_000, random_state=0)
treino_v, teste_v = amostra_v[amostra_v["ano"] < 2020], amostra_v[amostra_v["ano"] >= 2020]
m = modelos["árvores (gradient boosting)"]().fit(treino_v[X_cols + ["ret5_amanha"]], treino_v["alvo"])
auc_v = roc_auc_score(teste_v["alvo"], m.predict_proba(teste_v[X_cols + ["ret5_amanha"]])[:, 1])
mostra(f"   mesmo modelo com validação embaralhada (dias vizinhos no treino e no teste): AUC {np.mean(acc_emb):.3f}")
mostra(f"   com uma variável que olha o futuro (retorno dos 5 pregões seguintes): AUC {auc_v:.3f}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `ops_ifr2` da célula de abertura. Mostre o que muda com `sobrepor=False` (a variável `livre`) e por que contar cada sinal, com a operação anterior ainda aberta, cria sequências falsas no Z-score da Parte 9." Confira com a Parte 9, que imprime, no Ibovespa, 187 operações e Z-score de +0,38 com uma posição por vez, e 372 sinais, +0,59% por operação e Z-score de −5,25 contando cada sinal.
2. **Adaptar:** "Numa célula nova, abaixo da Parte 8, copie a Parte 8 e troque o ticker `PETR4.SA` por [ATIVO], uma das 40 ações do notebook. Se [ATIVO] não estiver entre elas, baixe-o à parte, com as mesmas datas e a função `limpa`, e guarde em `D`. Não mude `ops_ifr2` nem os 10.000 embaralhamentos. Qualquer conclusão sobre [ATIVO] precisa do teste fora da amostra (Capítulo 27)." Antes, rode o notebook na ordem (Executar tudo) e confira se a Parte 8 mostra, na PETR4, 131 operações, pior queda real de 53%, mediana de 55% e 5% piores de 66%, como no Gráfico 27.6.

Mais pedidos deste capítulo (entender e aplicar ao seu caso) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque a divisão da amostra (2006-2015 x 2016-2026) por 2006-2012 x 2013-2026. A melhor combinação de dentro continua ruim fora?
2. Na Parte 4, mude a janela do walk-forward de 5 para 3 anos. O resultado e os parâmetros escolhidos ficam mais instáveis?
3. Na Parte 5, troque o IFR(2) pelo Setup 9.1 do Capítulo 25. Ele passa no teste contra o acaso?
4. Na Parte 10, tire a purga de 10 dias entre o treino e o teste. A AUC muda?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.